# Reconstructing the  Experiments for the Submitted Paper

**Purpose:** Recover the experiments/results that were reported in the submitted paper but are not present in the currently recovered notebooks.

This notebook is written as a **reconstruction notebook**. It does not invent missing results. Wherever an exact historical artifact (for example, an annotation CSV or an old extraction log) is required, the notebook reports that dependency explicitly.

## Paper configuration to reproduce

- RegulationQA: 1,002 questions
- BGE: `BAAI/bge-base-en-v1.5`
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- BM25 + BGE + RRF
- RRF `k = 60`
- Per-retriever pool = 100
- Post-fusion CE candidate pool = **20**
- Sliding window = ±1 passage
- Inner score floor = **0.0**
- Inner decay tolerance = **5.0**

## Missing experiments reconstructed here

1. δin sensitivity sweep
2. Top-1 / query-guided / oracle best-of-3 span modes
3. BERTScore (RoBERTa-large)
4. Full-chunk vs extracted-span comparison
5. Sentence-selection vs contiguous-span comparison
6. RegulationQA window ablation under the paper configuration
7. DPDP Act → DPDP Rules retrieval-confusion statistic
8. ObliQA official-segmentation comparison
9. 50-case error-analysis extraction
10. Single-document vs cross-document breakdown
11. Qwen/Mistral evaluation from saved extraction logs
12. Hallucination/NLI analysis from saved generation outputs
13. Human-evaluation statistics from saved annotation data

> **Important:** A result in this notebook is only a reconstruction if the underlying artifact/data is actually available. The paper's published number is shown separately as an expected target, never silently substituted for a computed value.


In [ ]:
# Install dependencies (Kaggle/Colab friendly)
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk bert-score transformers accelerate bitsandbytes scikit-learn pandas

import os
import re
import json
import glob
import math
import string
import random
import gc
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import faiss
import nltk
from nltk.tokenize import sent_tokenize

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Dependencies loaded.")


In [ ]:
# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

# Primary artifact location used by the recovered Kaggle notebooks.
CANDIDATE_ARTIFACT_DIRS = [
    "/kaggle/input/regulationqa-artifacts/regulationqa_artifacts",
    "/kaggle/input/datasets/sauraviitjapdcl/regulationqa-artifacts",
    "/mnt/data"
]

ARTIFACT_DIR = next(
    (p for p in CANDIDATE_ARTIFACT_DIRS if os.path.exists(p)),
    CANDIDATE_ARTIFACT_DIRS[0]
)

OUTPUT_DIR = Path("/kaggle/working/paper_reconstruction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RRF_K = 60
PRE_FUSION_K = 100
POST_FUSION_M = 20
WINDOW_RADIUS = 1
TAU_IN = 0.0
DELTA_IN = 5.0

BGE_NAME = "BAAI/bge-base-en-v1.5"
CE_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Sensitivity subset: the historical notebook did not preserve the exact
# 340-question IDs. Set a saved list here when you recover it.
SENSITIVITY_N = 340
SENSITIVITY_SEED = 42

# Official-segmentation comparison uses a configurable 500-question sample
# because the submitted paper only states "a 500-question subsample".
OFFICIAL_SEGMENT_N = 500
OFFICIAL_SEGMENT_SEED = 42

print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print({
    "RRF_K": RRF_K,
    "PRE_FUSION_K": PRE_FUSION_K,
    "POST_FUSION_M": POST_FUSION_M,
    "WINDOW_RADIUS": WINDOW_RADIUS,
    "TAU_IN": TAU_IN,
    "DELTA_IN": DELTA_IN,
})


In [ ]:
# Locate known artifacts without assuming every file exists.
def find_artifact(*names):
    for base in CANDIDATE_ARTIFACT_DIRS:
        for name in names:
            p = Path(base) / name
            if p.exists():
                return str(p)
    return None

for name in [
    "chunks.pkl",
    "faiss_index.bin",
    "bm25.pkl",
    "regulationqa_v1.csv",
    "extraction_results.csv",
    "extraction_log.json",
    "bge_extraction_log.json",
    "bm25_extraction_log.json",
]:
    print(f"{name:<35} -> {find_artifact(name)}")


In [ ]:
# Load RegulationQA artifacts
chunks_path = find_artifact("chunks.pkl")
index_path = find_artifact("faiss_index.bin")
bm25_path = find_artifact("bm25.pkl")
rqa_path = find_artifact("regulationqa_v1.csv")

if not all([chunks_path, index_path, bm25_path, rqa_path]):
    raise FileNotFoundError(
        "RegulationQA artifacts are incomplete. Recover chunks.pkl, "
        "faiss_index.bin, bm25.pkl and regulationqa_v1.csv first."
    )

with open(chunks_path, "rb") as f:
    all_chunks = pickle.load(f)

index = faiss.read_index(index_path)

with open(bm25_path, "rb") as f:
    bm25 = pickle.load(f)

rqa_df = pd.read_csv(rqa_path)

print("Chunks:", len(all_chunks))
print("Questions:", len(rqa_df))
print("Columns:", list(rqa_df.columns))


In [ ]:
# Import pickle after the artifact-discovery cell.
import pickle

# Load models only when needed.
bge_model = SentenceTransformer(BGE_NAME)
cross_encoder = CrossEncoder(CE_NAME)

print("BGE and cross-encoder loaded.")


In [ ]:
# ------------------------------------------------------------------
# Core helpers, matching the recovered paper notebooks
# ------------------------------------------------------------------

def tokenize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()

def token_f1(prediction, ground_truth):
    # Matches the recovered notebook's set-based overlap formulation.
    pred = str(prediction).lower().split()
    gt = str(ground_truth).lower().split()
    common = set(pred) & set(gt)
    if not common:
        return 0.0
    p = len(common) / max(len(pred), 1)
    r = len(common) / max(len(gt), 1)
    return 2 * p * r / (p + r)

def get_doc_chunks(doc_name):
    return [
        (i, c) for i, c in enumerate(all_chunks)
        if c["doc_name"] == doc_name
    ]

def get_window_parts(doc_name, passage_idx, radius=WINDOW_RADIUS):
    doc_chunks = get_doc_chunks(doc_name)
    current_pos = next(
        (pos for pos, (i, c) in enumerate(doc_chunks) if i == passage_idx),
        None
    )
    if current_pos is None:
        c = all_chunks[passage_idx]
        return [(passage_idx, c["text"])]

    parts = []
    for offset in range(-radius, radius + 1):
        pos = current_pos + offset
        if 0 <= pos < len(doc_chunks):
            parts.append(doc_chunks[pos])
    return parts

def score_sentences(query, sentences):
    if not sentences:
        return []
    scores = cross_encoder.predict([[query, s] for s in sentences])
    return [(float(score), sent) for score, sent in zip(scores, sentences)]

def inner_decay_scored(scored_sentences, threshold=TAU_IN, delta=DELTA_IN):
    if not scored_sentences:
        return []

    scored = sorted(scored_sentences, key=lambda x: x[0], reverse=True)
    selected = []

    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))

    return selected if selected else [scored[0]]

pid_to_idx = {
    (c["doc_name"], c["chunk_idx"]): i
    for i, c in enumerate(all_chunks)
}

print("Helpers ready.")


In [ ]:
# ------------------------------------------------------------------
# Exact paper-configured RRF retrieval + detailed extraction
# ------------------------------------------------------------------

def rrf_candidates(query, pre_k=PRE_FUSION_K, post_k=POST_FUSION_M):
    q_emb = bge_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    _, bge_idx = index.search(q_emb, pre_k)
    bge_idx = bge_idx[0]

    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_idx = np.argsort(bm25_scores)[::-1][:pre_k]

    rrf = {}

    for rank, idx in enumerate(bge_idx):
        c = all_chunks[idx]
        pid = (c["doc_name"], c["chunk_idx"])
        rrf[pid] = rrf.get(pid, 0.0) + 1.0 / (RRF_K + rank + 1)

    for rank, idx in enumerate(bm25_idx):
        c = all_chunks[idx]
        pid = (c["doc_name"], c["chunk_idx"])
        rrf[pid] = rrf.get(pid, 0.0) + 1.0 / (RRF_K + rank + 1)

    return sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:post_k]

def retrieve_full_pipeline_detailed(query, post_k=POST_FUSION_M, use_window=True,
                                    delta=DELTA_IN):
    candidates = rrf_candidates(query, PRE_FUSION_K, post_k)
    out = []

    for pid, rrf_score in candidates:
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue

        c = all_chunks[idx]
        own_sentences = sent_tokenize(c["text"])
        if not own_sentences:
            continue

        own_scored = score_sentences(query, own_sentences)
        own_rank_score = max(s for s, _ in own_scored)

        if use_window:
            window_parts = get_window_parts(c["doc_name"], idx)
        else:
            window_parts = [(idx, c["text"])]

        window_text = " ".join(text for _, text in window_parts)
        window_sentences = sent_tokenize(window_text)
        window_scored = score_sentences(query, window_sentences)
        selected = inner_decay_scored(
            window_scored,
            threshold=TAU_IN,
            delta=delta
        )

        if selected:
            out.append({
                "doc_name": c["doc_name"],
                "chunk_idx": c["chunk_idx"],
                "rrf_score": float(rrf_score),
                "own_rank_score": float(own_rank_score),
                "window_text": window_text,
                "window_sentences": window_sentences,
                "window_scored": window_scored,
                "selected": selected,
                "selected_text": " ".join(s for _, s in selected),
            })

    # Paper ranks candidates by the maximum CE score from their OWN passage.
    out.sort(key=lambda x: x["own_rank_score"], reverse=True)
    return out

print("Exact m=20 / delta=5 retrieval is ready.")


## 1. RegulationQA span modes

The submitted paper distinguishes:

- **Top-1:** extracted span(s) from the CE-ranked top candidate
- **Query-guided:** highest CE-scoring sentence across the top 3 candidates, without using the reference answer
- **Oracle best-of-3:** best span among the top 3 candidates selected using ground-truth F1, used only as an upper bound

The current recovered `03_regulationqa_results.ipynb` collapses these into a best-of-top-3 F1 evaluator. The cells below separate them explicitly.


In [ ]:
def evaluate_span_modes(df, delta=DELTA_IN, limit=None):
    rows = []

    work_df = df if limit is None else df.iloc[:limit]

    for row_idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
        query = str(row["question"])
        gt = str(row["ground_truth_answer"])

        results = retrieve_full_pipeline_detailed(
            query,
            post_k=POST_FUSION_M,
            use_window=True,
            delta=delta,
        )

        if not results:
            rows.append({
                "row_idx": row_idx,
                "question": query,
                "ground_truth": gt,
                "top1_span": "",
                "query_guided_span": "",
                "oracle_best3_span": "",
                "top1_f1": 0.0,
                "query_guided_f1": 0.0,
                "oracle_best3_f1": 0.0,
                "top1_doc": "",
                "oracle_doc": "",
            })
            continue

        # Top-1: keep the selected sentences from the CE top-ranked candidate.
        top1 = results[0]
        top1_span = top1["selected_text"]
        top1_f1 = token_f1(top1_span, gt)

        # Query-guided: highest CE-scoring SELECTED sentence across top 3.
        guided_candidates = []
        for r in results[:3]:
            guided_candidates.extend([
                (score, sent, r["doc_name"])
                for score, sent in r["selected"]
            ])

        if guided_candidates:
            guided_score, guided_sent, guided_doc = max(
                guided_candidates, key=lambda x: x[0]
            )
            guided_span = guided_sent
            guided_f1 = token_f1(guided_span, gt)
        else:
            guided_span, guided_f1, guided_doc = "", 0.0, ""

        # Oracle: select the best span by reference F1 among top 3.
        oracle_span = ""
        oracle_f1 = -1.0
        oracle_doc = ""
        for r in results[:3]:
            for _, sent in r["selected"]:
                f1 = token_f1(sent, gt)
                if f1 > oracle_f1:
                    oracle_f1 = f1
                    oracle_span = sent
                    oracle_doc = r["doc_name"]

        if oracle_f1 < 0:
            oracle_f1 = 0.0

        rows.append({
            "row_idx": row_idx,
            "question": query,
            "ground_truth": gt,
            "top1_span": top1_span,
            "query_guided_span": guided_span,
            "oracle_best3_span": oracle_span,
            "top1_f1": top1_f1,
            "query_guided_f1": guided_f1,
            "oracle_best3_f1": oracle_f1,
            "top1_doc": top1["doc_name"],
            "oracle_doc": oracle_doc,
        })

    return pd.DataFrame(rows)

# Full evaluation can be expensive.
# Set RUN_FULL_RQA = True when you want the 1,002-question run.
RUN_FULL_RQA = False

span_eval = evaluate_span_modes(
    rqa_df,
    delta=DELTA_IN,
    limit=None if RUN_FULL_RQA else 50,
)

print(span_eval[[
    "top1_f1", "query_guided_f1", "oracle_best3_f1"
]].mean())


In [ ]:
def summarize_span_modes(df):
    out = pd.DataFrame({
        "Mode": ["Top-1", "Query-guided", "Oracle best-of-3"],
        "Token F1": [
            df["top1_f1"].mean(),
            df["query_guided_f1"].mean(),
            df["oracle_best3_f1"].mean(),
        ],
        "R@0.3": [
            (df["top1_f1"] >= 0.3).mean(),
            (df["query_guided_f1"] >= 0.3).mean(),
            (df["oracle_best3_f1"] >= 0.3).mean(),
        ],
        "R@0.5": [
            (df["top1_f1"] >= 0.5).mean(),
            (df["query_guided_f1"] >= 0.5).mean(),
            (df["oracle_best3_f1"] >= 0.5).mean(),
        ],
    })
    return out

print(summarize_span_modes(span_eval).to_string(index=False))

print("\nExpected paper targets (reference only):")
print("Top-1 F1 = 0.2994")
print("Query-guided F1 = 0.3083")
print("Oracle best-of-3 F1 = 0.3737")


## 2. δin sensitivity

The paper reports a 340-question sensitivity analysis over:

`0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0, 5.5, 20.0`

The exact historical 340-question IDs were not preserved in the recovered notebooks. The cell below therefore creates a deterministic 340-question sample unless a saved ID list is supplied.


In [ ]:
# Optional: recover the historical sensitivity subset if you later find it.
SENSITIVITY_IDS_PATH = find_artifact(
    "sensitivity_question_ids.csv",
    "regulationqa_sensitivity_ids.csv"
)

if SENSITIVITY_IDS_PATH:
    ids_df = pd.read_csv(SENSITIVITY_IDS_PATH)
    candidate_cols = [c for c in ids_df.columns if c in {"id", "question_id", "row_idx"}]
    if not candidate_cols:
        raise ValueError("Sensitivity ID file found, but no id/question_id/row_idx column was detected.")
    sens_ids = set(ids_df[candidate_cols[0]].tolist())
    if "id" in rqa_df.columns:
        sensitivity_df = rqa_df[rqa_df["id"].isin(sens_ids)].copy()
    else:
        sensitivity_df = rqa_df.loc[
            rqa_df.index.to_series().isin(sens_ids)
        ].copy()
    print("Using recovered sensitivity IDs:", len(sensitivity_df))
else:
    sensitivity_df = rqa_df.sample(
        n=min(SENSITIVITY_N, len(rqa_df)),
        random_state=SENSITIVITY_SEED
    ).copy()
    print(
        f"No historical ID list found. Using deterministic sample of "
        f"{len(sensitivity_df)} questions, seed={SENSITIVITY_SEED}."
    )

delta_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0, 5.5, 20.0]
sensitivity_rows = []

for delta in delta_values:
    print(f"\nRunning delta={delta} ...")
    res = evaluate_span_modes(
        sensitivity_df,
        delta=delta,
        limit=None
    )
    f1 = res["oracle_best3_f1"]
    sensitivity_rows.append({
        "delta": delta,
        "F1": f1.mean(),
        "R@0.3": (f1 >= 0.3).mean(),
        "R@0.5": (f1 >= 0.5).mean(),
    })

sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)

sensitivity_results.to_csv(
    OUTPUT_DIR / "delta_sensitivity.csv",
    index=False
)

print("\nExpected paper target at delta=5:")
print("F1=0.3696, R@0.3=0.4212, R@0.5=0.2884")


## 3. BERTScore

The submitted paper uses BERTScore F1 with `RoBERTa-large`. This section evaluates:

- top-1 extracted span
- query-guided sentence
- oracle best-of-3 span
- full retrieved chunk

The paper reports 0.8804, 0.8801, 0.8939 for the three full-pipeline span modes and ~0.815 for the full-chunk baselines.


In [ ]:
from bert_score import score as bertscore_score

def mean_bertscore(predictions, references):
    predictions = [str(x) if str(x).strip() else " " for x in predictions]
    references = [str(x) if str(x).strip() else " " for x in references]

    P, R, F1 = bertscore_score(
        predictions,
        references,
        model_type="roberta-large",
        lang="en",
        verbose=True,
        rescale_with_baseline=False,
    )
    return {
        "P": float(P.mean()),
        "R": float(R.mean()),
        "F1": float(F1.mean()),
    }

if len(span_eval) > 0:
    refs = span_eval["ground_truth"].tolist()

    bert_results = {
        "Top-1": mean_bertscore(span_eval["top1_span"], refs),
        "Query-guided": mean_bertscore(span_eval["query_guided_span"], refs),
        "Oracle best-of-3": mean_bertscore(span_eval["oracle_best3_span"], refs),
    }

    print(pd.DataFrame(bert_results).T)
else:
    bert_results = {}


In [ ]:
# Full-chunk comparison for BGE-only, BM25-only and full pipeline.
# This uses the top-1 retrieved chunk and does NOT use the reference answer
# to select a chunk.

def retrieve_bge_top1(query):
    q = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, idx = index.search(q, 1)
    c = all_chunks[int(idx[0][0])]
    return c["doc_name"], c["text"]

def retrieve_bm25_top1(query):
    scores = bm25.get_scores(tokenize(query))
    idx = int(np.argmax(scores))
    c = all_chunks[idx]
    return c["doc_name"], c["text"]

def build_full_chunk_comparison(df, limit=None):
    work_df = df if limit is None else df.iloc[:limit]
    rows = []

    for row_idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
        q = str(row["question"])
        gt = str(row["ground_truth_answer"])

        bge_doc, bge_chunk = retrieve_bge_top1(q)
        bm25_doc, bm25_chunk = retrieve_bm25_top1(q)

        fp = retrieve_full_pipeline_detailed(q, post_k=POST_FUSION_M)
        fp_top1 = fp[0]["selected_text"] if fp else ""

        rows.append({
            "row_idx": row_idx,
            "question": q,
            "ground_truth": gt,
            "bge_chunk": bge_chunk,
            "bm25_chunk": bm25_chunk,
            "pipeline_span": fp_top1,
            "bge_words": len(bge_chunk.split()),
            "bm25_words": len(bm25_chunk.split()),
            "pipeline_words": len(fp_top1.split()),
        })

    return pd.DataFrame(rows)

# Set True for full 1,002-question run.
RUN_FULL_CHUNK_COMPARISON = False

chunk_comparison = build_full_chunk_comparison(
    rqa_df,
    limit=None if RUN_FULL_CHUNK_COMPARISON else 50
)

refs = chunk_comparison["ground_truth"].tolist()

chunk_bert = {
    "BGE full chunk": mean_bertscore(
        chunk_comparison["bge_chunk"], refs
    ),
    "BM25 full chunk": mean_bertscore(
        chunk_comparison["bm25_chunk"], refs
    ),
    "Pipeline extracted": mean_bertscore(
        chunk_comparison["pipeline_span"], refs
    ),
}

print(pd.DataFrame(chunk_bert).T)

print("\nMean word counts:")
print(chunk_comparison[
    ["bge_words", "bm25_words", "pipeline_words"]
].mean())

chunk_comparison.to_csv(
    OUTPUT_DIR / "full_chunk_vs_extracted.csv",
    index=False
)

print("\nExpected paper references:")
print("BGE full chunk BERTScore ≈ 0.8150")
print("BM25 full chunk BERTScore ≈ 0.8151")
print("Pipeline top-1 BERTScore = 0.8804")
print("Pipeline span ≈ 54 words; full chunk ≈ 500 words")


## 4. Sentence selection vs contiguous span

The submitted paper reports:

- Sentence selection: F1 = 0.3696, BERT-F1 = 0.8939
- Contiguous span: F1 = 0.3925, BERT-F1 = 0.8540

The implementation below constructs a contiguous block from the first to last selected sentence in the original window order, so intervening sentences are included.


In [ ]:
def contiguous_from_selected(result):
    window_sentences = result["window_sentences"]
    selected_texts = {sent for _, sent in result["selected"]}

    positions = [
        i for i, sent in enumerate(window_sentences)
        if sent in selected_texts
    ]

    if not positions:
        return result["selected_text"]

    lo, hi = min(positions), max(positions)
    return " ".join(window_sentences[lo:hi+1])

def extraction_strategy_eval(df, limit=None):
    work_df = df if limit is None else df.iloc[:limit]
    rows = []

    for row_idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
        q = str(row["question"])
        gt = str(row["ground_truth_answer"])
        results = retrieve_full_pipeline_detailed(q)

        if not results:
            rows.append({
                "row_idx": row_idx,
                "ground_truth": gt,
                "sentence_span": "",
                "contiguous_span": "",
                "sentence_f1": 0.0,
                "contiguous_f1": 0.0,
            })
            continue

        # Oracle best-of-3 candidate under token F1, matching the
        # paper's 0.3696 / 0.8939 extraction-strategy comparison.
        best_result = None
        best_f1 = -1.0

        for r in results[:3]:
            candidate_f1 = token_f1(r["selected_text"], gt)
            if candidate_f1 > best_f1:
                best_f1 = candidate_f1
                best_result = r

        sentence_span = best_result["selected_text"]
        contiguous_span = contiguous_from_selected(best_result)

        rows.append({
            "row_idx": row_idx,
            "ground_truth": gt,
            "sentence_span": sentence_span,
            "contiguous_span": contiguous_span,
            "sentence_f1": token_f1(sentence_span, gt),
            "contiguous_f1": token_f1(contiguous_span, gt),
        })

    return pd.DataFrame(rows)

strategy_eval = extraction_strategy_eval(
    rqa_df,
    limit=None if RUN_FULL_RQA else 50
)

print(
    "Sentence selection F1:",
    strategy_eval["sentence_f1"].mean()
)
print(
    "Contiguous span F1:",
    strategy_eval["contiguous_f1"].mean()
)

refs = strategy_eval["ground_truth"].tolist()
print(
    "Sentence BERTScore:",
    mean_bertscore(strategy_eval["sentence_span"], refs)
)
print(
    "Contiguous BERTScore:",
    mean_bertscore(strategy_eval["contiguous_span"], refs)
)

strategy_eval.to_csv(
    OUTPUT_DIR / "sentence_vs_contiguous.csv",
    index=False
)

print("\nExpected paper references:")
print("Sentence selection: F1=0.3696, BERT-F1=0.8939")
print("Contiguous span: F1=0.3925, BERT-F1=0.8540")


## 5. Window ablation under the submitted configuration

This is deliberately separate from the older recovered ablation notebook because that notebook used `delta=2.0` in its helper function.

Here the configuration is exactly:

`m=20, tau=0.0, delta=5.0, window=±1`.


In [ ]:
def evaluate_window_condition(df, use_window=True, limit=None):
    work_df = df if limit is None else df.iloc[:limit]
    f1s, r03, r05 = [], [], []

    for _, row in tqdm(work_df.iterrows(), total=len(work_df)):
        q = str(row["question"])
        gt = str(row["ground_truth_answer"])

        results = retrieve_full_pipeline_detailed(
            q,
            post_k=POST_FUSION_M,
            use_window=use_window,
            delta=DELTA_IN
        )

        # Paper's window ablation was span-level and evaluated through
        # the best extracted span among the top three retrieved candidates.
        best_f1 = 0.0
        for r in results[:3]:
            f1 = token_f1(r["selected_text"], gt)
            if f1 > best_f1:
                best_f1 = f1

        f1s.append(best_f1)
        r03.append(float(best_f1 >= 0.3))
        r05.append(float(best_f1 >= 0.5))

    return {
        "F1": float(np.mean(f1s)),
        "R@0.3": float(np.mean(r03)),
        "R@0.5": float(np.mean(r05)),
    }

window_ablation_with = evaluate_window_condition(
    rqa_df,
    use_window=True,
    limit=None if RUN_FULL_RQA else 50
)

window_ablation_without = evaluate_window_condition(
    rqa_df,
    use_window=False,
    limit=None if RUN_FULL_RQA else 50
)

print("With window   :", window_ablation_with)
print("Without window:", window_ablation_without)

print("\nExpected paper references:")
print("With window    F1=0.3623 R@0.3=0.4062 R@0.5=0.2764")
print("Without window F1=0.3551 R@0.3=0.3962 R@0.5=0.2685")


## 6. DPDP Act → DPDP Rules retrieval confusion

The paper states that the DPDP Act and DPDP Rules have centroid cosine similarity 0.977 and that **27.5% of DPDP Act questions retrieve passages from the Rules instead**.

The recovered corpus-analysis notebook computes the centroid similarity graph, but not the 27.5% statistic. The calculation below derives the statistic from **top-1 retrieval**, not from an oracle/best-F1 span.


In [ ]:
def normalize_doc_name(name):
    s = str(name).lower()
    if "dpdp" in s and "rule" in s:
        return "DPDP Rules"
    if "dpdp" in s and "act" in s:
        return "DPDP Act"
    return str(name)

def dpdp_confusion_analysis(df, limit=None):
    work_df = df if limit is None else df.iloc[:limit]
    rows = []

    for _, row in tqdm(work_df.iterrows(), total=len(work_df)):
        source = normalize_doc_name(row.get("document_source", ""))
        if source != "DPDP Act":
            continue

        q = str(row["question"])
        results = retrieve_full_pipeline_detailed(q, post_k=POST_FUSION_M)

        top_doc = results[0]["doc_name"] if results else ""
        rows.append({
            "question": q,
            "gold_source": source,
            "top1_doc": top_doc,
            "top1_normalized": normalize_doc_name(top_doc),
            "is_rules_confusion": normalize_doc_name(top_doc) == "DPDP Rules",
        })

    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out, None

    return out, float(out["is_rules_confusion"].mean())

dpdp_rows, dpdp_rate = dpdp_confusion_analysis(
    rqa_df,
    limit=None if RUN_FULL_RQA else 50
)

if dpdp_rate is not None:
    print(f"Computed DPDP Act -> Rules top-1 confusion: {dpdp_rate:.4%}")
    display(dpdp_rows.head())
    dpdp_rows.to_csv(
        OUTPUT_DIR / "dpdp_rules_confusion.csv",
        index=False
    )
else:
    print("No DPDP Act rows were detected in the current sample.")

print("\nExpected paper reference: 27.5%")


## 7. ObliQA official-segmentation comparison

The submitted paper reports a 500-question subsample in which switching to official ObliQA segmentation recovered roughly +0.003–+0.004 MAP@10.

This cell rebuilds:

- the recovered paper's custom segmentation: long passages split to overlapping 1,000-character segments
- official source passages from the ObliQA repository, without the custom re-segmentation

Because the paper does not state the historical 500-question sampling seed, the seed is configurable.


In [ ]:
import requests

def load_obliqua_official():
    test_url = "https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_test.json"
    test = json.loads(requests.get(test_url, timeout=60).text)

    official = []
    for doc_id in range(1, 41):
        url = (
            "https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/"
            f"StructuredRegulatoryDocuments/{doc_id}.json"
        )
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            continue
        official.extend(r.json())

    return test, official

# Uncomment when internet access is available.
# obliqua_test, obliqua_official_passages = load_obliqua_official()

print("This section requires internet access to the public ObliQA repository.")


In [ ]:
def build_obliqua_custom_passages(official_passages):
    clean = []

    for p in official_passages:
        text = p["Passage"].strip()

        if len(text) < 30:
            continue

        if len(text) > 1500:
            start = 0
            while start < len(text):
                chunk = text[start:start+1000]
                if len(chunk) < 100 and start > 0:
                    break

                clean.append({
                    "DocumentID": p["DocumentID"],
                    "PassageID": p["PassageID"],
                    "Passage": chunk
                })
                start += 800
        else:
            clean.append({
                "DocumentID": p["DocumentID"],
                "PassageID": p["PassageID"],
                "Passage": text
            })

    return clean

def map_answer_ground_truth(item):
    return {
        (p["DocumentID"], p["PassageID"])
        for p in item["Passages"]
    }

def eval_obliqua_passage_retrieval(test_items, passages, sample_seed=42):
    # This function is intended to be run after the official dataset is loaded.
    texts = [p["Passage"] for p in passages]
    emb = bge_model.encode(
        texts,
        batch_size=128,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    faiss_index = faiss.IndexFlatIP(emb.shape[1])
    faiss_index.add(emb)

    bm25_local = BM25Okapi([tokenize(t) for t in texts])

    def bge_top10(q):
        qe = bge_model.encode(
            [q], normalize_embeddings=True,
            convert_to_numpy=True
        ).astype(np.float32)
        _, idx = faiss_index.search(qe, 10)
        return idx[0].tolist()

    def bm25_top10(q):
        sc = bm25_local.get_scores(tokenize(q))
        return np.argsort(sc)[::-1][:10].tolist()

    sample = test_items
    if len(test_items) > OFFICIAL_SEGMENT_N:
        rng = np.random.default_rng(sample_seed)
        chosen = rng.choice(
            len(test_items),
            size=OFFICIAL_SEGMENT_N,
            replace=False
        )
        sample = [test_items[int(i)] for i in chosen]

    def evaluate_fn(retrieve_fn):
        r10 = []
        map10 = []

        for item in tqdm(sample):
            gt = map_answer_ground_truth(item)
            retrieved = retrieve_fn(item["Question"])

            pids = [
                (passages[i]["DocumentID"], passages[i]["PassageID"])
                for i in retrieved
            ]

            r10.append(
                float(any(pid in gt for pid in pids[:10]))
            )

            ap = 0.0
            hits = 0
            for rank, pid in enumerate(pids[:10], 1):
                if pid in gt:
                    hits += 1
                    ap += hits / rank

            map10.append(ap / max(len(gt), 1))

        return {
            "R@10": float(np.mean(r10)),
            "MAP@10": float(np.mean(map10)),
            "n": len(sample)
        }

    return {
        "BGE official": evaluate_fn(bge_top10),
        "BM25 official": evaluate_fn(bm25_top10),
    }

print("Official-segmentation evaluation function ready.")


## 8. 50-case error-analysis extraction

The paper says 50 failure cases were examined and four patterns were identified.

The missing historical annotation itself is not recoverable from the current notebooks, so this section produces a reproducible **50-case evidence sheet** rather than inventing the historical labels.

The sheet contains objective fields:

- whether the correct source appears in the top-20 candidate pool
- top-1 document
- top-3 best F1
- top-1 F1
- extraction gap
- question/document metadata
- top candidate texts/spans

A `manual_category` column is included for recovered historical annotations.


In [ ]:
def build_50_case_error_sheet(df):
    rows = []

    # Use full data if available; find actual top-1 failures.
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        q = str(row["question"])
        gt = str(row["ground_truth_answer"])

        results = retrieve_full_pipeline_detailed(q, post_k=POST_FUSION_M)

        if not results:
            continue

        top1_f1 = token_f1(results[0]["selected_text"], gt)

        best3_f1 = 0.0
        best3_span = ""
        for r in results[:3]:
            for _, sent in r["selected"]:
                f1 = token_f1(sent, gt)
                if f1 > best3_f1:
                    best3_f1 = f1
                    best3_span = sent

        if top1_f1 >= best3_f1 and top1_f1 >= 0.5:
            continue

        candidate_docs = [r["doc_name"] for r in results[:POST_FUSION_M]]
        source = str(row.get("document_source", ""))

        rows.append({
            "row_idx": idx,
            "question": q,
            "document_source": source,
            "ground_truth": gt,
            "top1_doc": results[0]["doc_name"],
            "top1_span": results[0]["selected_text"],
            "top1_f1": top1_f1,
            "best3_span": best3_span,
            "best3_f1": best3_f1,
            "extraction_gap": best3_f1 - top1_f1,
            "source_in_top20": source in candidate_docs,
            "candidate_docs_top20": " | ".join(candidate_docs),
            "manual_category": "",
            "manual_notes": "",
        })

        if len(rows) >= 50:
            break

    out = pd.DataFrame(rows)
    out.to_csv(
        OUTPUT_DIR / "error_analysis_50_cases.csv",
        index=False
    )

    return out

# Set ERROR_SAMPLE_FULL = True for the full run.
ERROR_SAMPLE_FULL = False

if ERROR_SAMPLE_FULL:
    error_50 = build_50_case_error_sheet(rqa_df)
    display(error_50.head())
else:
    print("Set ERROR_SAMPLE_FULL=True to generate the 50-case evidence sheet.")


## 9. Single-document vs cross-document breakdown

The paper states that cross-document questions perform substantially worse because retrieval is performed independently, without an evidence-assembly stage.

This notebook does not assume a particular dataset schema. It first looks for an explicit cross-document indicator and otherwise reports the available metadata so the historical labeling can be restored.


In [ ]:
def find_cross_doc_column(df):
    candidates = [
        "is_cross_document",
        "cross_document",
        "is_multidoc",
        "multi_document",
        "question_type",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None

cross_col = find_cross_doc_column(rqa_df)
print("Detected cross-document field:", cross_col)

if cross_col:
    if rqa_df[cross_col].dtype == bool:
        cross_mask = rqa_df[cross_col]
    else:
        s = rqa_df[cross_col].astype(str).str.lower()
        cross_mask = s.str.contains(
            "cross|multi|comparison|complement|gap|combined"
        )

    single_df = rqa_df[~cross_mask]
    cross_df = rqa_df[cross_mask]

    print("Single-document rows:", len(single_df))
    print("Cross-document rows :", len(cross_df))
    print(
        "\nNo metrics are printed here until you run the full retrieval "
        "evaluation; this cell only establishes the subset."
    )
else:
    print(
        "No explicit cross-document indicator found. "
        "Recover the historical metadata/labels before computing this claim."
    )


## 10. Qwen3-4B / Mistral-7B evaluation from saved extraction logs

The recovered LLM notebook already contains the judge implementation. This section is retained here so the full reconstruction notebook has one place to regenerate the model-based evaluation.

It requires the saved extraction logs:
- `extraction_log.json`
- `bge_extraction_log.json`
- `bm25_extraction_log.json`


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import cohen_kappa_score

LLM_PROMPT = '''You are evaluating a regulatory information retrieval system.

QUESTION: {question}

REFERENCE ANSWER: {ground_truth}

EXTRACTED SPAN: {extracted_span}

Does the extracted span correctly answer the question?
Consider it correct even if the wording differs from the reference,
as long as the information is accurate and relevant.

Respond with exactly one of:
2 - Fully correct: span contains the complete answer
1 - Partially correct: span contains relevant but incomplete information
0 - Incorrect: span does not answer the question

Your response (single digit only):'''

def score_with_model(model, tokenizer, device, question, ground_truth, span):
    if not span or not str(span).strip():
        return 0

    prompt = LLM_PROMPT.format(
        question=question,
        ground_truth=ground_truth,
        extracted_span=span
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    del inputs, outputs, new_tokens
    torch.cuda.empty_cache()
    gc.collect()

    for char in response:
        if char in ["0", "1", "2"]:
            return int(char)
    return 0

def run_llm_eval(log, model, tokenizer, device):
    scores = []
    for item in tqdm(log):
        scores.append(
            score_with_model(
                model,
                tokenizer,
                device,
                item["question"],
                item["ground_truth"],
                item.get("best_span", "")
            )
        )

    a = np.asarray(scores)
    return {
        "fully_correct": float((a == 2).mean()),
        "partially_correct": float((a == 1).mean()),
        "incorrect": float((a == 0).mean()),
        "correct_partial": float((a >= 1).mean()),
        "mean_score": float(a.mean()),
        "scores": scores,
    }

print("LLM evaluation code ready.")


In [ ]:
# Load saved extraction logs when available.
log_paths = {
    "Full pipeline": find_artifact("extraction_log.json"),
    "BGE only": find_artifact("bge_extraction_log.json"),
    "BM25 only": find_artifact("bm25_extraction_log.json"),
}

print(log_paths)

loaded_logs = {}
for name, path in log_paths.items():
    if path:
        with open(path, "r", encoding="utf-8") as f:
            loaded_logs[name] = json.load(f)
        print(name, len(loaded_logs[name]))

if not loaded_logs:
    print("No extraction logs found. Recover them before running the LLM evaluation.")


## 11. Hallucination / NLI analysis

The paper reports a 100-question experiment comparing generated answers from:

- full retrieved chunk (~496 words)
- extracted spans (~54 words)

using Qwen3-4B and DeBERTa-v3 NLI, with:

- Faithfulness: 14.7% → 43.1%
- Neutral: 71.7% → 44.1%
- Contradiction: 13.6% → 12.8%

The missing prerequisite is the historical **generated-answer file**. The cell below accepts a CSV/JSON with one row per question and columns for the two generated answers and the corresponding contexts.


In [ ]:
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"

def classify_nli_sentences(
    answers,
    contexts,
    model_name=NLI_MODEL_NAME
):
    nli = CrossEncoder(model_name)

    labels = []
    label_names = ["contradiction", "entailment", "neutral"]

    for answer, context in tqdm(
        zip(answers, contexts),
        total=len(answers)
    ):
        sents = sent_tokenize(str(answer))
        if not sents:
            continue

        scores = nli.predict([
            [context, sent] for sent in sents
        ])

        scores = np.asarray(scores)
        for row in scores:
            labels.append(
                label_names[int(np.argmax(row))]
            )

    return labels

def summarize_nli(labels):
    counts = Counter(labels)
    total = max(sum(counts.values()), 1)
    return {
        "faithfulness_entailment_pct": 100 * counts["entailment"] / total,
        "neutral_pct": 100 * counts["neutral"] / total,
        "contradiction_pct": 100 * counts["contradiction"] / total,
        "n": total,
    }

print("NLI utilities ready.")
print("Expected paper reference:")
print("Faithfulness 14.7% -> 43.1%")
print("Neutral      71.7% -> 44.1%")
print("Contradict   13.6% -> 12.8%")


## 12. Human evaluation

The submitted paper reports 531 manually annotated spans, two annotators, and Cohen/weighted kappa values.

No historical annotation table was included in the recovered notebooks. This section therefore expects a recovered annotation file rather than reconstructing labels.

Recommended long-format schema:

`question_id, condition, annotator, score`

where `condition ∈ {top1, best_of_3}` and `score ∈ {0, 0.5, 1}`.


In [ ]:
from sklearn.metrics import cohen_kappa_score

def human_eval_summary(df):
    required = {"condition", "annotator", "score"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required human-eval columns: {sorted(missing)}"
        )

    annotators = sorted(df["annotator"].unique())
    if len(annotators) != 2:
        raise ValueError(
            f"Expected exactly two annotators, found {len(annotators)}"
        )

    a1, a2 = annotators
    rows = []

    for condition in sorted(df["condition"].unique()):
        sub = df[df["condition"] == condition].copy()
        pivot = sub.pivot_table(
            index=[c for c in ["question_id"] if c in sub.columns],
            columns="annotator",
            values="score",
            aggfunc="first"
        ).dropna()

        if a1 not in pivot.columns or a2 not in pivot.columns:
            continue

        x = pivot[a1].astype(float).to_numpy()
        y = pivot[a2].astype(float).to_numpy()

        # For the paper's unweighted kappa, map 0.5 to a separate class.
        kappa = cohen_kappa_score(x, y, labels=[0.0, 0.5, 1.0])

        # A simple linearly-weighted implementation for 0/0.5/1.
        cats = np.array([0.0, 0.5, 1.0])
        weight_matrix = np.abs(
            cats[:, None] - cats[None, :]
        )
        # sklearn does not accept continuous labels with weighted kappa
        # in a stable way across versions, so encode classes explicitly.
        encode = {0.0: 0, 0.5: 1, 1.0: 2}
        xi = np.array([encode[v] for v in x])
        yi = np.array([encode[v] for v in y])
        weighted_kappa = cohen_kappa_score(
            xi, yi, weights="linear"
        )

        rows.append({
            "condition": condition,
            "n_pairs": len(pivot),
            "annotator1_correct_or_partial_pct": 100 * (x >= 0.5).mean(),
            "annotator2_correct_or_partial_pct": 100 * (y >= 0.5).mean(),
            "combined_mean_score": float((x.mean() + y.mean()) / 2),
            "cohen_kappa": float(kappa),
            "linear_weighted_kappa": float(weighted_kappa),
        })

    return pd.DataFrame(rows)

human_path = find_artifact(
    "human_annotations.csv",
    "human_eval.csv",
    "annotations.csv"
)

if human_path:
    human_df = pd.read_csv(human_path)
    human_summary = human_eval_summary(human_df)
    display(human_summary)
else:
    print("No historical human-annotation CSV found.")
    print("Expected references:")
    print("Best-of-3: annotator partial/correct 65.0% and 48.4%; kappa=0.711; weighted=0.798")
    print("Top-1    : annotator partial/correct 65.0% and 44.8%; kappa=0.648; weighted=0.739")


## 13. Save a reconstruction manifest

This records which source artifacts were actually available when the reconstruction was run. It is useful when comparing an old result with the number reported in the submitted paper.


In [ ]:
manifest = {
    "artifact_dir": ARTIFACT_DIR,
    "configuration": {
        "rrf_k": RRF_K,
        "pre_fusion_k": PRE_FUSION_K,
        "post_fusion_m": POST_FUSION_M,
        "window_radius": WINDOW_RADIUS,
        "tau_in": TAU_IN,
        "delta_in": DELTA_IN,
        "bge": BGE_NAME,
        "cross_encoder": CE_NAME,
    },
    "available_artifacts": {
        name: find_artifact(name)
        for name in [
            "chunks.pkl",
            "faiss_index.bin",
            "bm25.pkl",
            "regulationqa_v1.csv",
            "extraction_results.csv",
            "extraction_log.json",
            "bge_extraction_log.json",
            "bm25_extraction_log.json",
            "human_annotations.csv",
            "human_eval.csv",
            "annotations.csv",
        ]
    },
    "paper_reference_targets": {
        "obliqua_full_recall10": 0.8029,
        "obliqua_full_map10": 0.5915,
        "rqa_top1_f1": 0.2994,
        "rqa_top1_bertscore": 0.8804,
        "rqa_query_guided_f1": 0.3083,
        "rqa_query_guided_bertscore": 0.8801,
        "rqa_best3_f1": 0.3737,
        "rqa_best3_bertscore": 0.8939,
        "window_with_f1": 0.3623,
        "window_without_f1": 0.3551,
        "delta5_f1": 0.3696,
        "dpdp_rules_confusion_pct": 27.5,
        "hallucination_faithfulness_full_pct": 14.7,
        "hallucination_faithfulness_extracted_pct": 43.1,
        "human_best3_kappa": 0.711,
        "human_top1_kappa": 0.648,
    }
}

with open(OUTPUT_DIR / "reconstruction_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nSaved:", OUTPUT_DIR / "reconstruction_manifest.json")
